# Cartesian Pendulum Clean Example

Minimal notebook for the current workflow: sample contexts with `sample_X_shake_pulse`, simulate with `simulate_cartesian_pendulum_custom`, and evaluate safety with `check_cartesian_safety`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve().parents[0]
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data.cartesian_pendulum import (
    CartesianPendulumParams,
    check_cartesian_safety,
    pivot_shake_pulse_batch,
    sample_X_shake_pulse,
    simulate_cartesian_pendulum_custom,
)


In [ ]:
params = CartesianPendulumParams(mass=1.0, length=1.0, gravity=9.81)

num_traj = 1000
T = 10.0
dt = 0.005
t0 = T / 2.0
theta_threshold_deg = 35.0
time_window = (3.0, 7.0)
seed = 21


In [ ]:
X = sample_X_shake_pulse(
    n_samples=num_traj,
    A_distribution='gaussian',
    A_loc=0.10,
    A_scale=0.02,
    bound_A=(0.0, 0.2),
    omega_distribution='uniform',
    omega_range=(1.0, 3.0),
    tau_range=(0.45, 1.2),
    t0=t0,
    q=(1.0, 0.0, 0.0),
    a0=(0.0, 0.0, 0.0),
    seed=seed,
)

traj = simulate_cartesian_pendulum_custom(
    params=params,
    t_final=T,
    dt=dt,
    X=X,
    pivot_eval=pivot_shake_pulse_batch,
)


In [ ]:
safety_indicator = check_cartesian_safety(
    traj,
    threshold=theta_threshold_deg,
    window=time_window,
)

p_safe = float(np.mean(safety_indicator))
print(f'Empirical safety probability: {p_safe:.4f}')
print(f'Safe trajectories: {int(np.sum(safety_indicator))}/{len(safety_indicator)}')

plt.figure(figsize=(9, 4))
for i in range(min(40, traj['theta_deg'].shape[0])):
    color = 'tab:blue' if safety_indicator[i] else 'crimson'
    alpha = 0.35 if safety_indicator[i] else 0.6
    plt.plot(traj['t'], traj['theta_deg'][i], color=color, alpha=alpha, linewidth=1.0)

plt.axhline(theta_threshold_deg, color='black', linestyle='--', linewidth=1.2, label='threshold')
plt.axvspan(*time_window, color='grey', alpha=0.12, label='safety window')
plt.xlabel('Time')
plt.ylabel('Angle from vertical (deg)')
plt.title('Cartesian pendulum trajectories')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
